In [13]:
import open3d as o3d
import numpy as np
import math
from scipy.spatial.transform import Rotation as R


def default_visualization(object, zoom=1.0):
    azimuth_deg = -45
    elevation_deg = -135

    az = math.radians(azimuth_deg)
    el = math.radians(elevation_deg)

    # spherical → cartesian (camera front vector)
    front = np.array([
    math.cos(el) * math.cos(az),
    math.cos(el) * math.sin(az),
    math.sin(el)])

    # Open3D camera looks toward object
    front = -front
    lookat = [0, 0, 0]
    up = [0, 0, 1]  # typical CAD Z-up

    o3d.visualization.draw_geometries(
        object,
        window_name="Default Visualization",
        width=1024,
        height=768,
        lookat=lookat,
        up=up,
        front=front,
        zoom=zoom
    )

def get_euler_from_matrix(R):
    """Extracts Roll, Pitch, Yaw from a 3x3 Rotation Matrix"""
    # Using the standard XYZ convention
    pitch = math.asin(-R[2, 0])
    roll = math.atan2(R[2, 1], R[2, 2])
    yaw = math.atan2(R[1, 0], R[0, 0])
    
    # Return as degrees for easier reading
    return [math.degrees(roll), math.degrees(pitch), math.degrees(yaw)]

def pose_to_matrix(pose):
    """
    Converts [x, y, z, roll, pitch, yaw] in degrees to a 4x4 transformation matrix.
    If zyz=True, assumes the input is in ZYZ Euler Angles, otherwise XYZ Euler Angles.
    
    Args:
    - pose (list or array): [x, y, z, roll, pitch, yaw] in degrees.
    - zyz (bool): If True, interprets the last three values as ZYZ Euler angles. 
                  If False, uses XYZ Euler angles (default).
    
    Returns:
    - T (ndarray): The 4x4 homogeneous transformation matrix.
    """
    x, y, z = pose[:3]   # Translation vector
    roll, pitch, yaw = pose[3:]  # Rotation angles in degrees
    
    # Convert XYZ Euler Angles to a 3x3 rotation matrix
    rot_matrix = R.from_euler('zyx', [roll, pitch, yaw], degrees=True).as_matrix()

    # Build the 4x4 Homogeneous Transformation Matrix
    T = np.eye(4)  # Start with the identity matrix (4x4)
    T[:3, :3] = rot_matrix  # Set the upper-left 3x3 part to the rotation matrix
    T[:3, 3] = [x, y, z]    # Set the upper-right 3x1 part to the translation vector
    return T

def inspect_viewpoint(target_idx, viewpoints_6d, viewpoints_visualization, mesh=None):

    sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.25)
    sphere.paint_uniform_color([1, 0, 0])

    selected_camera = viewpoints_visualization[target_idx]

    cam_pos = np.array(viewpoints_6d[target_idx][:3])

    print(f"\n--- Inspecting Viewpoint Index: {target_idx} ---")
    print(f"Camera Position (6D): {viewpoints_6d[target_idx]}")

    return sphere, selected_camera, cam_pos

def camera_pov_visualization(
        geoms,
        camera_pose,
        show_world_frame=False,
        show_camera_frame=False,
        save=0,
        index=0):

    vis = o3d.visualization.Visualizer()
    if save:
        vis.create_window(visible=False)
    else:
        vis.create_window()

    if show_world_frame:
        world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=50)
        vis.add_geometry(world_frame)

    for g in geoms:
        vis.add_geometry(g)

    ctr = vis.get_view_control()

    # --- Extract camera pose ---
    cam_pos = camera_pose[:3, 3]
    R_cam = camera_pose[:3, :3]

    # Optional camera frame visualization
    if show_camera_frame:
        cam_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=5)
        cam_frame.rotate(R_cam, center=(0, 0, 0))
        cam_frame.translate(cam_pos)
        vis.add_geometry(cam_frame)

    # --- Build extrinsic (world → camera) ---
    extrinsic = np.eye(4)
    extrinsic[:3, :3] = R_cam.T
    extrinsic[:3, 3] = -R_cam.T @ cam_pos

    param = ctr.convert_to_pinhole_camera_parameters()
    param.extrinsic = extrinsic
    # ctr.convert_from_pinhole_camera_parameters(param)
    ctr.convert_from_pinhole_camera_parameters(param, allow_arbitrary=True)

    vis.poll_events()
    vis.update_renderer()

    if save == 1:
        vis.capture_screen_image(f"viewpoints_candidate/viewpoint_object_{index}.png")
    elif save == 2:
        vis.capture_screen_image(f"viewpoints_candidate/viewpoint_pcd_{index}.png")
    else:
        vis.run()

    vis.destroy_window()

def hidden_point_removal_from_camera(pcd, camera_pose, radius_scale=5):
    """
    Returns visible-point cloud using Open3D HPR.
    Uses full 4x4 camera pose.
    """

    cam_pos = camera_pose[:3, 3]

    center = pcd.get_center()
    d_cam = np.linalg.norm(cam_pos - center)
    radius = d_cam * radius_scale

    _, pt_map = pcd.hidden_point_removal(cam_pos, radius)

    visible_pcd = pcd.select_by_index(pt_map)
    visible_pcd.paint_uniform_color([0, 1, 0])

    return visible_pcd


def generate_viewpoints(obj_center, count, height, diameter, intrinsic_z_rot=0):
    """
    Look-at rotation followed by an intrinsic Z-axis rotation.
    intrinsic_z_rot: degrees to spin the camera around its own lens axis.
    """
    poses = []
    radius = diameter / 2

    for i in range(count):
        azimuth = 2 * math.pi * i / count

        # 1. Position
        tx = obj_center[0] + radius * math.cos(azimuth)
        ty = obj_center[1] + radius * math.sin(azimuth)
        tz = obj_center[2] + height

        cam_pos = np.array([tx, ty, tz])
        target = np.array(obj_center)

        # 2. Basic Look-At Matrix
        z_axis = target - cam_pos
        z_axis = z_axis / np.linalg.norm(z_axis)
        up = np.array([0, 0, 1])
        x_axis = np.cross(z_axis, up)
        x_axis = x_axis / np.linalg.norm(x_axis)
        y_axis = np.cross(z_axis, x_axis)
        
        R_look_at = np.stack([x_axis, y_axis, z_axis], axis=1)

        # 3. Apply Intrinsic Rotation on Z
        # We create a rotation of 'intrinsic_z_rot' around the local Z [0,0,1]
        intrinsic_z_rot = 360/count * i - 90
        local_rot = R.from_euler('z', intrinsic_z_rot, degrees=True)
        
        # Combine: R_final = R_look_at * R_local
        # In SciPy, multiplying Rotation objects handles the matrix math correctly
        r_combined = R.from_matrix(R_look_at) * local_rot

        # 4. Convert to Euler 'zyx'
        rz, ry, rx = r_combined.as_euler('zyx', degrees=True)

        pose = [
            float(tx), float(ty), float(tz),
            float(rz), float(ry), float(rx)
        ]
        poses.append(pose)

    return poses



In [5]:
# Visualization setup

SOURCE_PATH = "workpiece/first_object.stl" #  CAD model

visualization = []

mesh = o3d.io.read_triangle_mesh(SOURCE_PATH)
mesh.compute_vertex_normals() # Makes the shading look better
# R_mat = mesh.get_rotation_matrix_from_axis_angle([0, 0, np.radians(90)])
# mesh.rotate(R_mat, center=[0, 0, 0])

pcd_all = o3d.geometry.PointCloud()
pcd_all = mesh.sample_points_poisson_disk(number_of_points=5000)
pcd_all.paint_uniform_color([0.25, 0, 0])
# visualization.append(pcd_all)

world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=50, origin=[0, 0, 0])
visualization.append(world_frame)


default_visualization([mesh], zoom=1.0)

In [14]:
viewpoints_6d = []
viewpoints_visualization = []

In [15]:
OBJ_POS = [0, 0, 0]
VIEWPOINT_COUNT = 12
SCAN_HEIGHT = 403.2
DESIRED_ANGLE = 30 # degrees from vertical

# Turn on the generation of viewpoints around the object
viewpoints_6d = []
viewpoints_visualization = []

SCAN_DIAMETER = 2 * (SCAN_HEIGHT * math.tan(math.radians(DESIRED_ANGLE)))
viewpoints_6d = generate_viewpoints(OBJ_POS, VIEWPOINT_COUNT, SCAN_HEIGHT, SCAN_DIAMETER)

# viewpoints_6d.extend(generate_viewpoints(OBJ_POS, VIEWPOINT_COUNT, SCAN_HEIGHT, SCAN_DIAMETER))

for vp in viewpoints_6d:

    cam_pos = np.array(vp[:3])

    rz, ry, rx = vp[3], vp[4], vp[5]
    R_mat = R.from_euler('zyx', [rz, ry, rx], degrees=True).as_matrix()

    cam_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=50)

    cam_frame.rotate(R_mat, center=(0,0,0))
    cam_frame.translate(cam_pos)

    viewpoints_visualization.append(cam_frame)


default_visualization(viewpoints_visualization + [mesh], zoom=1.5)


In [16]:
# default_visualization(viewpoints_visualization + [mesh], zoom=1.5)

print(viewpoints_6d)

[[232.7876285372571, 0.0, 403.2, -180.0, -30.000000000000004, 180.0], [201.6, 116.39381426862853, 403.2, 176.3099324740202, -25.658906273255287, 163.89788624801398], [116.39381426862857, 201.59999999999997, 403.2, 176.565051177078, -14.477512185929921, 153.43494882292202], [1.425413120846275e-14, 232.7876285372571, 403.2, -180.0, 0.0, 149.99999999999997], [-116.39381426862849, 201.6, 403.2, -176.56505117707798, 14.477512185929921, 153.434948822922], [-201.6, 116.39381426862853, 403.2, -176.30993247402023, 25.6589062732553, 163.897886248014], [-232.7876285372571, 2.85082624169255e-14, 403.2, -180.0, 29.999999999999993, 180.0], [-201.60000000000002, -116.39381426862847, 403.2, 176.30993247402023, 25.658906273255276, -163.897886248014], [-116.39381426862865, -201.5999999999999, 403.2, 176.56505117707803, 14.477512185929934, -153.434948822922], [-4.276239362538824e-14, -232.7876285372571, 403.2, -180.0, 0.0, -149.99999999999997], [116.39381426862857, -201.59999999999997, 403.2, -176.565051

In [17]:
# Check the visualization of one viewpoint in detail, including the camera position, target point, and the line connecting them.

check_all = False
index = 0


if not check_all:

    camera_pose = pose_to_matrix(viewpoints_6d[index])

    default_visualization(
        [mesh, viewpoints_visualization[index], world_frame],
        zoom=1.25
    )

    # Camera POV view
    camera_pov_visualization(
        [mesh],
        camera_pose,
        show_world_frame=False,
        show_camera_frame=True
    )

else:

    for index in range(len(viewpoints_6d)):

        camera_pose = pose_to_matrix(viewpoints_6d[index])

        camera_pov_visualization(
            [mesh],
            camera_pose,
            show_world_frame=False,
            show_camera_frame=True
        )


In [18]:
# Remove hidden points from the camera's perspective using Open3D's Hidden Point Removal (HPR) algorithm

# Extract camera position from camera_pose
cam_pos = camera_pose[:3, 3]

# Compute radius based on camera position
radius_scale=5
center = pcd_all.get_center()
d_cam = np.linalg.norm(cam_pos - center)
radius = d_cam * radius_scale

# Perform Hidden Point Removal (HPR) based on the camera position
visible_pcd = hidden_point_removal_from_camera(pcd_all, camera_pose, radius)

# Visualize the result
camera_pov_visualization(
    [visible_pcd],
    camera_pose,
    index=index
)


In [19]:
# Generate viewpoints_6dfor all points and save them to a file
print("Final feasible viewpoints_6dcount:", len(viewpoints_6d))

for i in range(len(viewpoints_6d)):

    # --- Build camera pose ---
    camera_pose = pose_to_matrix(viewpoints_6d[i])
    cam_pos = camera_pose[:3, 3]   # only needed for radius computation


    # Save simulated IMAGE
    camera_pov_visualization(
        [mesh],
        camera_pose,
        save=1,
        index=i
    )

    # Save simulated PCD
    radius_scale = 5
    center = pcd_all.get_center()
    d_cam = np.linalg.norm(cam_pos - center)
    radius = d_cam * radius_scale

    # Pass the full camera pose to the HPR function
    visible_pcd = hidden_point_removal_from_camera(
        pcd_all,
        camera_pose,     # Pass the full camera pose instead of just cam_pos
        radius
    )

    camera_pov_visualization(
        [visible_pcd],
        camera_pose,
        save=2,
        index=i
    )

    o3d.io.write_point_cloud(
        f"viewpoints_candidate/viewpoint_simulated_{i}.pcd",
        visible_pcd
    )

    # ==============================
    # Save 4x4 pose
    # ==============================
    np.save(
        f"viewpoints_candidate/viewpoint_pose_{i}.npy",
        camera_pose
    )

# Example check
output = np.load("viewpoints_candidate/viewpoint_pose_0.npy")
print("\nExample of output file transformation:\n", output)

Final feasible viewpoints_6dcount: 12

Example of output file transformation:
 [[-8.66025404e-01  1.06057524e-16 -5.00000000e-01  2.32787629e+02]
 [ 1.83697020e-16  1.00000000e+00 -1.06057524e-16  0.00000000e+00]
 [ 5.00000000e-01 -1.83697020e-16 -8.66025404e-01  4.03200000e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
